In [9]:
import pyvisa
import time

# === Configuration ===
channel = 3  # 1=CH1 (6V), 2=CH2 (+25V), 3=CH3 (−25V)
voltage = 3.3
current_limit = 0.2
output_duration = 5  # seconds

channel_names = {1: "CH1", 2: "CH2", 3: "CH3"}

try:
    # Initialize VISA connection
    rm = pyvisa.ResourceManager()
    resources = rm.list_resources()
    print("Available VISA resources:")
    for r in resources:
        print(" -", r)
    instrument = rm.open_resource("USB0::0x2A8D::0x1202::MY59003914::0::INSTR")  # Replace with your instrument ID
    instrument.timeout = 5000

    print("Connected to:", instrument.query("*IDN?").strip())

    # Select output channel
    instrument.write(f"INST:NSEL {channel}")

    # Set voltage and current limit
    instrument.write(f"VOLT {voltage}")
    instrument.write(f"CURR {current_limit}")

    # Turn on output
    instrument.write("OUTP ON")
    time.sleep(1)

    # Read back values
    set_voltage = instrument.query("VOLT?").strip()
    set_current = instrument.query("CURR?").strip()
    meas_voltage = instrument.query("MEAS:VOLT?").strip()
    meas_current = instrument.query("MEAS:CURR?").strip()

    # Format for display
    display_text = (
        f"Channel: {channel_names[channel]}\n"
        f"Vset={float(set_voltage):.5f}V Iset={float(set_current):.2f}A\n"
        f"Vout={float(meas_voltage):.5f}V Iout={float(meas_current):.2f}A"
    )

    print("Displayed:\n" + display_text)

    # Wait before turning off
    time.sleep(output_duration)

finally:
    try:
        instrument.write("OUTP OFF")
        instrument.close()
        print("Output OFF and connection closed.")
    except Exception as e:
        print("Shutdown error:", e)


Available VISA resources:
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12
Displayed:
Channel: CH3
Vset=3.30000V Iset=0.20A
Vout=3.29801V Iout=0.00A
Output OFF and connection closed.
